In [1]:
import sys
import os

# Add project root to Python path (works in notebooks)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
from RAG.retriever import Retrieval
from RAG.generator import Generator
from Evaluation.evaluation import Eval_Pipeline
import torch
import gc

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langsmith import Client
client = Client()


In [5]:
print(client.list_projects())

<generator object Client.list_projects at 0x000001F9857CA8F0>


In [6]:
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MiB")
print(f"Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MiB")


retriever = Retrieval()
generator = Generator()

test_data = [
{"query": "What is RAG in machine learning?", "expected": "RAG (Retrieval-Augmented Generation) is a framework that combines retrieval of external documents with generation by a language model."}]
#{"query": "Explain transformer models.", "expected": "Transformer models are neural networks that use self-attention mechanisms to process sequential data efficiently."},
#{"query": "What are embeddings in NLP?", "expected": "Embeddings are vector representations of words, sentences, or documents used to capture semantic meaning."},
#]


pipeline = Eval_Pipeline(retriever, generator)
df = pipeline.run(test_data)

GPU memory cleared
Allocated: 0.00 MiB
Cached: 0.00 MiB

📦 Loading model meta-llama/Llama-3.2-1B-Instruct on cuda...


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\accelerate\utils\modeling.py:821: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)


✅ Model loaded successfully.
EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=1)


c:\Users\user\Documents\RAG_Fact_Checking\Evaluation\metric_runner.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--microsoft--Phi-3-mini-128k-instruct-onnx. Caching files will still work but in a degraded

TypeError: not a string

In [ ]:
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MiB")
print(f"Cached: {torch.cuda.memory_reserved() / 1024**2:.2f} MiB")


retriever = Retrieval()
generator = Generator()

GPU memory cleared
Allocated: 0.00 MiB
Cached: 0.00 MiB

📦 Loading model meta-llama/Llama-3.2-1B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!
C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\accelerate\utils\modeling.py:821: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)


✅ Model loaded successfully.


In [ ]:
from Evaluation.llm_judge import LLMJudge
test_data = [
{"query": "What is RAG in machine learning?", "expected": "RAG (Retrieval-Augmented Generation) is a framework that combines retrieval of external documents with generation by a language model."}]
#{"query": "Explain transformer models.", "expected": "Transformer models are neural networks that use self-attention mechanisms to process sequential data efficiently."},
#{"query": "What are embeddings in NLP?", "expected": "Embeddings are vector representations of words, sentences, or documents used to capture semantic meaning."},
#]

judge=LLMJudge()
scores=[]
scores.extend(
    judge.evaluate(
        _["query"],
        generator.generate(query=_["query"], context=retriever.retrieve(_["query"])),_["expected"]
    )
    for _ in test_data
)

📦 Loading judge model 'meta-llama/Llama-3.2-1B-Instruct' on cuda...
✅ Judge model loaded successfully.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
print(scores)

['\nEvaluate the generated answer compared to the reference.\n\nCriteria:\n- Relevance\n- Faithfulness\n- Completeness\n- Fluency\n\nQuestion: What is RAG in machine learning?\nGenerated: RAG is a framework for building and training large language models. It is designed to provide a scalable and efficient way to train and evaluate language models, making it easier to develop and deploy large language models in various applications.\n\nExplanation:\n\nRAG stands for "Resilient Architecture for General Language Models". It is a framework that allows researchers and practitioners to build and train large language models, which are complex artificial intelligence models that can process and understand human language. RAG provides a set of tools and techniques for building, training, and evaluating language models, making it easier to develop and deploy large language models in various applications.\n\nRAG is based on the idea of building a general-purpose language model that can handle a w